In [ ]:
from __future__ import annotations
import os
import sys
from pathlib import Path
from datetime import datetime, timedelta, timezone
import time
import sys
from typing import Dict

import pandas as pd
import requests
from dotenv import load_dotenv

from clients.yahoo_finance_client import YahooFinanceClient
from ticker_mapping import TICKER_MAPPING
from utils.misc import load_existing_latest_by_symbol, merge_into_output
from utils.telegram import send_telegram_log


In [ ]:
def main() -> None:
    load_dotenv()
    ohlcv_output_path: Path = Path("../../data/yahoo_1m_all_symbols.parquet")
    ohlcv_output_path.parent.mkdir(parents=True, exist_ok=True)
    
        
    now_utc = datetime.now(timezone.utc).replace(second=0, microsecond=0) - timedelta(minutes=1)
    bootstrap_start = now_utc - timedelta(days=6)

    client = YahooFinanceClient()

    ohlcv_latest_by_symbol = load_existing_latest_by_symbol(ohlcv_output_path)

    temp_ohlcv_updates_path = ohlcv_output_path.with_suffix(".updates.tmp.parquet")


    ohlcv_all_updates: list[pd.DataFrame] = []
    
    tickers = [info['ticker'] for info in TICKER_MAPPING.values()]
    print(f"Found {len(tickers)} tickers. Output: {ohlcv_output_path}")
    
    print(bootstrap_start, now_utc)
    df_fetched = client.download_ohlc_list(tickers, "1m", bootstrap_start, now_utc)
    filter_df_list = []
    
    for ticker in tickers:
        
        ohlcv_latest_ts = ohlcv_latest_by_symbol.get(ticker, None)
        df_ticker = df_fetched[df_fetched["ticker"] == ticker].reset_index(names='timestamp').copy()
        
        if ohlcv_latest_ts is None:
            filter_df_list.append(df_ticker)
        else:
            filter_df_list.append(df_ticker[df_ticker["timestamp"] > ohlcv_latest_ts])
        
    if filter_df_list:
        ohlcv_all_updates = pd.concat(filter_df_list, ignore_index=True)
        ohlcv_all_updates = ohlcv_all_updates.drop_duplicates(subset=["ticker", "timestamp"]).sort_values(
            ["ticker", "timestamp"]
        )
        
        if ohlcv_output_path.exists():
            ohlcv_all_updates.to_parquet(temp_ohlcv_updates_path, index=False)
            print(f"Staged updates parquet: {temp_ohlcv_updates_path} ({len(ohlcv_all_updates):,} rows)")
            merge_into_output(ohlcv_output_path, temp_ohlcv_updates_path)
            temp_ohlcv_updates_path.unlink(missing_ok=True)
            print(f"Merged staged updates into: {ohlcv_output_path}")
        else:
            ohlcv_all_updates.to_parquet(ohlcv_output_path, index=False)
            print(f"Created new parquet: {ohlcv_output_path} ({len(ohlcv_all_updates):,} rows)")
    else:
        print("No new OHLCV rows to update!")


    tg_token = os.getenv("EQUITY_DATA_BOT_TOKEN")
    tg_chat_id = os.getenv("EQUITY_DATA_CHANNEL_ID")
    if tg_token and tg_chat_id:
        send_telegram_log(ohlcv_output_path, "Equity OHLCV Latest Timestamps:", tg_token, tg_chat_id)

In [ ]:
main()